# 01 · Statcast pitch-level cleaning

This notebook turns raw Statcast pitches (2021 to 2025, via `pybaseball`) into one clean table of every pitch thrown by qualifying starting pitchers, with each pitch standardized against that pitcher's own training-season baseline for that pitch type.

**Rules**
- Seasons 2021 to 2025; skip 2020. 2023 on is the pitch-clock era.
- Starters with at least 15 starts in a season.
- Train 2021 to 2023, validate 2024, test 2025, split by whole starts.
- Baselines and standardization use **training seasons only**.
- Missing spin or tracking values are kept as `NaN` and flagged, so the models can drop them or marginalize them out.

## Setup

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pybaseball import statcast, cache

cache.enable()  # pybaseball caches each day it downloads, so an interrupted pull resumes

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())  # repo root, wherever the notebook runs
RAW = ROOT / "data/raw/statcast"
CLEAN = ROOT / "data/clean"
FIGS = ROOT / "figures"
for p in (RAW, CLEAN, FIGS):
    p.mkdir(parents=True, exist_ok=True)

SEASONS = [2021, 2022, 2023, 2024, 2025]  # 2020 skipped (60-game season)
SPLIT = {2021: "train", 2022: "train", 2023: "train", 2024: "val", 2025: "test"}
MIN_STARTS = 15      # starts per season to qualify (brief: about 120 to 150 pitchers per year)
MIN_PITCHES = 30     # pitches needed for a start to count; drops openers (see step 3)
MIN_BASELINE = 50    # training pitches needed for a pitcher-specific baseline (TODO: tune on train only)

# Emission features (brief: release, physics, movement, location)
FEATURES = ["release_speed", "release_spin_rate", "spin_axis",
            "release_pos_x", "release_pos_z", "release_extension",
            "pfx_x", "pfx_z", "plate_x", "plate_z"]

# Columns kept from the raw pull
COLS = ["game_pk", "game_date", "game_type", "home_team", "away_team",
        "inning", "inning_topbot", "outs_when_up", "bat_score", "fld_score",
        "pitcher", "player_name", "p_throws", "batter", "stand",
        "at_bat_number", "pitch_number", "n_thruorder_pitcher", "balls", "strikes",
        "pitch_type", "description", "type", "events",
        "launch_speed", "launch_angle", "delta_run_exp"] + FEATURES

# Thrown pitches that should not be emissions: pitchout, intentional ball, unknown, generic "FA"
NON_EMISSION_TYPES = ["PO", "IN", "UN", "FA"]

## Pull raw Statcast (cached)

One parquet file per season.

The date range is wider than the regular season on purpose; spring training and postseason games are dropped in step 2 using `game_type`.

In [4]:
for year in SEASONS:
    path = RAW / f"statcast_{year}.parquet"
    if path.exists():
        continue
    raw_year = statcast(start_dt=f"{year}-03-01", end_dt=f"{year}-11-30")
    raw_year.to_parquet(path)  # full raw columns, so nothing has to be re-pulled later
    print(year, raw_year.shape)

This is a large query, it may take a moment to complete
Skipping offseason dates
Skipping offseason dates


100%|██████████| 246/246 [00:14<00:00, 16.86it/s]


2021 (765733, 119)
This is a large query, it may take a moment to complete
Skipping offseason dates
Skipping offseason dates


100%|██████████| 246/246 [00:14<00:00, 16.61it/s]


2022 (775330, 119)
This is a large query, it may take a moment to complete
Skipping offseason dates
Skipping offseason dates


100%|██████████| 246/246 [00:14<00:00, 16.50it/s]


2023 (774038, 119)
This is a large query, it may take a moment to complete
Skipping offseason dates
Skipping offseason dates


100%|██████████| 246/246 [00:15<00:00, 15.95it/s]


2024 (760249, 119)
This is a large query, it may take a moment to complete
Skipping offseason dates
Skipping offseason dates


100%|██████████| 246/246 [00:15<00:00, 15.72it/s]


2025 (770795, 119)


## Load, keep regular-season pitches, sort

In [5]:
raw = pd.concat([pd.read_parquet(RAW / f"statcast_{y}.parquet", columns=COLS) for y in SEASONS],
                ignore_index=True)

df = raw[raw.game_type == "R"].copy()
df["game_date"] = pd.to_datetime(df.game_date)
df["season"] = df.game_date.dt.year
df = df[df.season.isin(SEASONS)]

# Statcast occasionally repeats a pitch; one row per (game, plate appearance, pitch)
df = df.drop_duplicates(["game_pk", "at_bat_number", "pitch_number"])

# Raw pulls come in reverse order; put every game in pitch order
df = df.sort_values(["game_pk", "at_bat_number", "pitch_number"]).reset_index(drop=True)
print(f"{len(raw):,} raw rows -> {len(df):,} regular-season pitches")

3,846,145 raw rows -> 3,567,641 regular-season pitches


## Keep only starting pitchers

The starter for a team is whoever threw that team's first pitch of the game. The fielding team is the home team in the top of an inning and the away team in the bottom.

**Openers.** An opener (a reliever who starts and throws an inning) does not produce a fatigue sequence, so starts with fewer than `MIN_PITCHES = 30` pitches are dropped. In the 2021 to 2025 pull this removes 87 of 18,826 starts (0.5%) among pitchers with 15+ starts, and most are early injury exits rather than true openers, so the rule also removes a few very short injury exits. That is acceptable because 30 pitches is too short to show fatigue, but note that the shortest pulls are excluded (Informative censoring). The filter is applied before the 15-start qualification, so a pitcher's start count excludes these games.

TO DO: Record this choice in the Data section of the report.

In [6]:
df["fld_team"] = np.where(df.inning_topbot == "Top", df.home_team, df.away_team)
starter = df.groupby(["game_pk", "fld_team"]).pitcher.first().rename("starter")
df = df.join(starter, on=["game_pk", "fld_team"])
df = df[df.pitcher == df.starter].drop(columns="starter")

# Pitch-clock violations create automatic balls/strikes with no thrown pitch; they are not pitches
df = df[~df.description.isin(["automatic_ball", "automatic_strike"])]

df["start_id"] = df.game_pk.astype(str) + "_" + df.pitcher.astype(str)

# Drop openers and other very short outings
n_pitch = df.groupby("start_id").start_id.transform("size")
print(f"Dropping {(n_pitch < MIN_PITCHES).groupby(df.start_id).first().sum():,} starts under {MIN_PITCHES} pitches")
df = df[n_pitch >= MIN_PITCHES]
print(f"{df.start_id.nunique():,} starts, {len(df):,} pitches")

Dropping 591 starts under 30 pitches
23,705 starts, 2,044,904 pitches


## 4. Qualifying starters (≥ 15 starts in a season)

Applied season by season, so a pitcher can qualify in 2022 and not in 2023.

Note for the Data section: qualifying in the test season uses that season's start count, which is information from the future. It only defines *who* is in the sample, not any model input, but say so in the report.

In [ ]:
## Qualifying starters (≥ 15 starts in a season)

Applied season by season, so a pitcher can qualify in 2022 and not in 2023.

Note for the Data section: qualifying in the test season uses that season's start count, which is information from the future. It only defines *who* is in the sample, not any model input, but say so in the report.

## 5. Sequence variables and split labels

- `pitch_idx`: pitch count within the start (1, 2, ...), the fatigue baseline's main input.
- `n_thruorder_pitcher` from Statcast is times through the order (TTO).
- `last_pitch`: the pitch after which the manager pulled the starter (informative censoring).
- `split` follows the season, so every start lands wholly in one split.

In [ ]:
## Sequence variables and split labels

- `pitch_idx`: pitch count within the start (1, 2, ...), the fatigue baseline's main input.
- `n_thruorder_pitcher` from Statcast is times through the order (TTO).
- `last_pitch`: the pitch after which the manager pulled the starter (informative censoring).
- `split` follows the season, so every start lands wholly in one split.

## 6. Missing data and unusable pitches

Rows are **kept** so that each start stays a complete sequence (pitch count stays correct). Two flags say what each row can be used for:

- `emission`: a real pitch with a pitch-type label (not a pitchout, intentional ball or unknown).
- `complete`: all emission features present. Use this to drop rows, or ignore it and marginalize the missing features out of the HMM emission density.

TODO: check MLB pitch-type labels with unsupervised clustering in analysis.

In [ ]:
## Missing data and unusable pitches

Rows are **kept** so that each start stays a complete sequence (pitch count stays correct). Two flags say what each row can be used for:

- `emission`: a real pitch with a pitch-type label (not a pitchout, intentional ball or unknown).
- `complete`: all emission features present. Use this to drop rows, or ignore it and marginalize the missing features out of the HMM emission density.

TODO: check MLB pitch-type labels with unsupervised clustering in analysis.

## 7. Baselines from training seasons only, then standardize

For each pitcher and pitch type, the mean and standard deviation of every feature over **2021 to 2023 only**. Every pitch (train, validation and test) becomes `z = (x - mean) / sd`, so a slider is compared with that pitcher's own sliders, not with his fastball.

Fallback: a pitcher-pitch type pair with fewer than `MIN_BASELINE` training pitches (including pitchers who debut in 2024 or 2025) uses the league training baseline for that pitch type and throwing hand. The `baseline` column records which was used. TODO: decide whether to keep those pitchers or drop them, and state it in the report.

Note: `spin_axis` is an angle in degrees. A linear z-score is fine as long as a pitch type does not straddle 0/360; check this.

In [ ]:
## Baselines from training seasons only, then standardize

For each pitcher and pitch type, the mean and standard deviation of every feature over **2021 to 2023 only**. Every pitch (train, validation and test) becomes `z = (x - mean) / sd`, so a slider is compared with that pitcher's own sliders, not with his fastball.

Fallback: a pitcher-pitch type pair with fewer than `MIN_BASELINE` training pitches (including pitchers who debut in 2024 or 2025) uses the league training baseline for that pitch type and throwing hand. The `baseline` column records which was used. TODO: decide whether to keep those pitchers or drop them, and state it in the report.

Note: `spin_axis` is an angle in degrees. A linear z-score is fine as long as a pitch type does not straddle 0/360; check this.

## 8. Sanity check (training seasons only)

Average standardized velocity by pitch count. A gentle decline is the fatigue trend the latent models must beat. Only training data is plotted so that exploring the data never looks at validation or test starts.

In [ ]:
## Sanity check (training seasons only)

Average standardized velocity by pitch count. A gentle decline is the fatigue trend the latent models must beat. Only training data is plotted so that exploring the data never looks at validation or test starts.

## Save

In [12]:
df.to_parquet(CLEAN / "statcast_starters.parquet", index=False)

starts = df.groupby("start_id").agg(game_pk=("game_pk", "first"), pitcher=("pitcher", "first"),
                                    player_name=("player_name", "first"), p_throws=("p_throws", "first"),
                                    game_date=("game_date", "first"), season=("season", "first"),
                                    split=("split", "first"), n_pitches=("n_pitches_start", "first"),
                                    max_tto=("n_thruorder_pitcher", "max")).reset_index()
starts.to_parquet(CLEAN / "starts.parquet", index=False)

baselines = pd.concat([mu.add_suffix("_mean"), sd.add_suffix("_sd"), n_train.rename("n_train")],
                      axis=1, join="inner").reset_index()
baselines.to_parquet(CLEAN / "baselines.parquet", index=False)
print(df.shape, starts.shape, baselines.shape)

(1654821, 58) (18669, 10) (1182, 23)


## Summary of saved dataframes

All files are in `data/clean/`.

### `statcast_starters.parquet`: one row per pitch (1,661,964 rows × 58 columns)
Every regular-season pitch thrown by a qualifying starter (≥ 15 starts in that season), in pitch order within each game.
- **Raw Statcast columns kept:** game and player ids, teams, inning, outs, score, batter and handedness, count, `pitch_type`, `description`, `events`, batted-ball fields, and the 10 emission `FEATURES` (release speed, spin rate, spin axis, release position x/z, extension, `pfx_x`, `pfx_z`, `plate_x`, `plate_z`).
- **Added sequence columns:** `season`, `start_id` (`game_pk_pitcher`), `pitch_idx` (pitch count within the start), `n_pitches_start`, `last_pitch` (the pitch after which the starter was pulled), `split` (train 2021 to 2023, val 2024, test 2025), `clock_era` (2023 onward).
- **Flags:** `emission` (a real labeled pitch; 99.9% of rows) and `complete` (all 10 features present; 99.6% of rows).
- **Standardized features:** `z_<feature>` for each of the 10 features. Each is a z-score against that pitcher's own training-season baseline for that pitch type. It is `NaN` where `emission` is false or the raw value is missing.
- **`baseline`:** `pitcher` or `league`, which baseline the z-score used. Training pitches are almost all pitcher-specific (3,137 league). Validation has 92,329 league-baseline pitches and test has 118,174, mostly pitchers who debuted after 2023.

| split | starts | pitchers | pitches |
|---|---|---|---|
| train | 11,247 | 244 | 990,918 |
| val | 3,812 | 150 | 338,817 |
| test | 3,767 | 148 | 332,229 |

### `starts.parquet`: one row per start (18,826 rows × 10 columns)
Columns: `start_id`, `game_pk`, `pitcher`, `player_name`, `p_throws`, `game_date`, `season`, `split`, `n_pitches` (total pitches in the start), `max_tto` (most times through the order reached). The `game_pitcher_data` notebook uses this as its input and joins weather and pitcher info onto it.

### `baselines.parquet`: one row per pitcher and pitch type (1,188 rows × 23 columns)
The training-season (2021 to 2023) mean (`<feature>_mean`) and standard deviation (`<feature>_sd`) of each of the 10 features, plus `n_train`, the number of training pitches. Only pairs with at least `MIN_BASELINE = 50` training pitches are kept (244 pitchers; median 579 pitches per pair). Pairs that fall below the minimum use the league baseline by throwing hand and pitch type, which is not saved here.